# Datenreduktion: Qualität vs. Input

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** Wie viel spektraler Input ist für die Klassifikation von Kultur und Stadium nötig (#155, Meilenstein 3)? Wir variieren den Input des besten Modells (kombinierte RBF-SVM) und messen die Qualität per Kreuzvalidierung auf dem Trainingsteil.

**Vorgehen:**

1. Feature-Audit: Welche Merkmalsgruppen tragen überhaupt bei?
2. Trade-off-Kurve über 17 Reduktionsvarianten (PCA, Bandauswahl, jedes n-te Band, Sentinel-2, nur Indizes, nur Meta).
3. Empfehlung nach einer festen CV-Regel, danach **eine** Validierung.

Zwei Maße für den Input:

- **Gemessene Bänder:** was ein Sensor messen müsste. PCA braucht weiter alle Bänder.
- **Merkmale im Modell:** was die SVM sieht.

Alle Läufe stehen im MLflow-Experiment `band-reduction`.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from awp2.config import (
    MLFLOW_REDUCTION_EXPERIMENT,
    MODEL_DOCS_FIGURES_DIR,
    N_INPUT_FEATURES_COL,
    N_MEASURED_BANDS_COL,
    REDUCTION_BEST_PARAMS_COL,
    REDUCTION_FAMILY_COL,
    REDUCTION_MAX_CV_LOSS,
    REDUCTION_TUNING_RUN_COL,
    REDUCTION_VARIANT_COL,
)
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models.combined import make_combined_svm
from awp2.plots import plot_metric_comparison, plot_tradeoff, save_doc_figure
from awp2.preprocessing import PreprocessingConfig
from awp2.reduction import CURVE_METRICS, reduction_curve, reduction_variants

## 1. Referenz und Suchraum

Referenz ist die kombinierte RBF-SVM auf standardisierten Spektren mit AEZ und zyklischem Monat (`combined_svm_cv_reference`, CV-BAcc kombiniert 0,845). Jede Variante bekommt dasselbe kleine Raster aus `C` und `gamma`. `gamma="scale"` passt sich der Merkmalszahl an, denn das beste `gamma` hängt von der Zahl der Merkmale ab.

In [ ]:
APPROACH = "combined"
model = make_combined_svm()
base = PreprocessingConfig(use_spectral_standard_scale=True)
grid = {
    "estimator__C": [30.0, 100.0, 300.0],
    "estimator__gamma": [0.001, 0.003, 0.01, 0.03, "scale"],
}
variants = reduction_variants(base)
pd.DataFrame({"Variante": [v.name for v in variants], "Familie": [v.family for v in variants]}).T

## 2. Feature-Audit: welche Gruppen tragen bei?

Ausgehend von der Referenz wird je eine Merkmalsgruppe weggelassen oder ergänzt. Jede Zeile ist ein eigenes Tuning (CV, kein Validierungsteil).

In [ ]:
audit_configs = {
    "Spektren + Meta (Referenz)": base,
    "Nur Spektren": base.model_copy(update={"use_meta": False}),
    "+ Vegetationsindizes": base.model_copy(update={"use_vegetation_indices": True}),
    "+ Mittlere Albedo": base.model_copy(update={"use_mean_albedo": True}),
    "+ PC_1/PC_2": base.model_copy(update={"use_principal_components": True}),
    "Nur Indizes + Meta": PreprocessingConfig(
        use_spectral_standard_scale=True, use_spectra=False, use_vegetation_indices=True
    ),
    "Nur Meta": PreprocessingConfig(use_spectra=False),
}
audit_slugs = ["reference", "no_meta", "indices", "albedo", "pcs", "indices_meta", "meta"]
audit_rows = []
for (label, preprocessing), slug in zip(audit_configs.items(), audit_slugs, strict=True):
    tuned = tune(
        model,
        TuneConfig(
            name=f"audit_svm_{slug}",
            approach=APPROACH,
            experiment=MLFLOW_REDUCTION_EXPERIMENT,
            description=f"Feature-Audit: {label}",
            param_grid=grid,
            preprocessing=preprocessing,
        ),
    )
    best = tuned.candidates.loc[tuned.candidates["rank"].idxmin()]
    audit_rows.append(
        {
            "Merkmale": label,
            "CV-BAcc kombiniert": tuned.best_score,
            "Std": best["std"],
            **tuned.input_size._asdict(),
        }
    )
audit = pd.DataFrame(audit_rows).set_index("Merkmale")
audit.round(3)

In [ ]:
audit_ax = (
    audit["CV-BAcc kombiniert"]
    .sort_values()
    .plot.barh(xerr=audit["Std"], figsize=(8, 4), capsize=3)
)
audit_ax.axvline(
    audit.loc["Spektren + Meta (Referenz)", "CV-BAcc kombiniert"], color="grey", linestyle="--"
)
audit_ax.set_xlabel("CV-BAcc kombiniert")
audit_ax.set_title("Feature-Audit der kombinierten SVM")
save_doc_figure(audit_ax.figure, "datenreduktion_audit", MODEL_DOCS_FIGURES_DIR)

**Befund Feature-Audit:**

- **AEZ und Monat sind unverzichtbar:** Ohne sie fällt die CV-BAcc kombiniert von 0,848 auf 0,777 (−0,07). Allein schaffen sie nur 0,537. Spektrum und Kontext ergänzen sich also, keines ersetzt das andere.
- **Vegetationsindizes (0,839), mittlere Albedo (0,849) und PC_1/PC_2 (0,842) bringen nichts:** Alle Unterschiede liegen innerhalb einer Standardabweichung (≈ 0,02). Die SVM holt dieselbe Information schon aus den Rohbändern; die drei Gruppen bleiben aus dem Modell.
- **Nur Indizes + Meta** liest 28 Bänder und kommt auf 0,829. Fachwissen allein trägt weit, verliert aber 0,02 gegenüber dem vollen Spektrum.

## 3. Trade-off-Kurve

`reduction_curve` tunt jede Variante und bewertet die beste Einstellung je Fold für Kultur, Stadium und kombiniert. Punkte zeigen das Mittel, Fehlerbalken die Standardabweichung über die fünf Folds.

In [ ]:
curve = reduction_curve(
    model,
    variants,
    TuneConfig(
        name="reduction_svm",
        approach=APPROACH,
        experiment=MLFLOW_REDUCTION_EXPERIMENT,
        description="Datenreduktionsstudie: Variante der Eingabe, gleiches Raster",
        param_grid=grid,
        preprocessing=base,
    ),
)
curve.drop(columns=[REDUCTION_BEST_PARAMS_COL, REDUCTION_TUNING_RUN_COL]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, metric in zip(axes, CURVE_METRICS, strict=True):
    plot_tradeoff(curve, metric=metric, ax=ax)
    ax.set_title(ax.get_ylabel())
fig.suptitle("Qualität vs. gemessene Bänder")
fig.tight_layout()
save_doc_figure(fig, "datenreduktion_tradeoff_baender", MODEL_DOCS_FIGURES_DIR)

**Befund Trade-off (gemessene Bänder):**

- **Breite Abdeckung schlägt die Auswahl einzelner Bänder:**
    - Sentinel-2 misst nur 10 breite Bänder und erreicht 0,852, gleichauf mit allen 131 Bändern (0,848).
    - Die überwachte Bandauswahl braucht dafür 80 Bänder (0,839). Mit 10 Bändern kommt sie nur auf 0,758.
    - Grund: Die univariate Auswahl greift benachbarte, fast identische SWIR2-Bänder. Sentinel-2 verteilt seine Bänder über VIS, Red Edge, NIR und beide SWIR-Bereiche.
- **Die feine Hyperion-Auflösung wird kaum gebraucht:**
    - Jedes 4. Band (33 Bänder) erreicht 0,855.
    - Jedes 8. Band (17 Bänder) erreicht noch 0,845.
- **Sentinel-2 vor jedem 8. Band:** Vermutlich mittelt jedes breite Band über mehrere Hyperion-Bänder und glättet so das Rauschen.
- **Kultur und Stadium verhalten sich gleich:** Die reduzierten Eingaben verlieren bei keinem der beiden Ziele mehr als bei der kombinierten Metrik.
- **Die Spitzengruppe ist statistisch gleichauf:** Alle Bänder, PCA-20/40, jedes 2./4./8. Band und Sentinel-2 liegen zwischen 0,845 und 0,855, bei einer Standardabweichung von ≈ 0,02 über die Folds. Belegt ist „kein Verlust“, nicht „besser“.

In [ ]:
features_ax = plot_tradeoff(curve, x=N_INPUT_FEATURES_COL)
features_ax.set_title("Qualität vs. Merkmale im Modell")
save_doc_figure(features_ax.figure, "datenreduktion_tradeoff_merkmale", MODEL_DOCS_FIGURES_DIR)

**Befund Merkmale im Modell:**

- **PCA verkleinert das Modell, nicht den Messaufwand:** PCA-20 gibt der SVM nur 29 statt 140 Merkmale bei gleicher Qualität (0,853), braucht dafür aber weiterhin alle 131 Bänder. Für „Dateninput minimieren“ ist PCA deshalb keine Lösung.
- **Unter 10 Komponenten fällt PCA deutlich ab:** PCA-5 erreicht 0,810, PCA-2 0,760. Dass zwei Komponenten über 90 % der Varianz erklären (EDA), heißt nicht, dass sie die trennende Information enthalten.
- **Sentinel-2 ist auch auf dieser Achse das kleinste Modell ohne Verlust:** 19 Merkmale, davon 10 Spektralbänder.

## 4. Empfehlung nach CV-Regel

Die Regel steht vor dem Blick auf die Validierung fest: Gesucht ist die Variante mit den **wenigsten gemessenen Bändern**, die höchstens `REDUCTION_MAX_CV_LOSS` (0,01) CV-BAcc kombiniert gegenüber allen Bändern verliert. Zusätzlich prüfen wir Sentinel-2 als praktisch verfügbaren Sensor.

In [ ]:
reference = curve.set_index(REDUCTION_VARIANT_COL).loc["all_bands"]
equivalent = curve[
    (curve["bacc_combined"] >= reference["bacc_combined"] - REDUCTION_MAX_CV_LOSS)
    & (curve[REDUCTION_FAMILY_COL] != "all_bands")
]
recommended = equivalent.nsmallest(1, N_MEASURED_BANDS_COL)
chosen = ["all_bands", *recommended[REDUCTION_VARIANT_COL], "sentinel2"]
chosen = list(dict.fromkeys(chosen))
equivalent[[REDUCTION_VARIANT_COL, N_MEASURED_BANDS_COL, "bacc_combined"]].sort_values(
    N_MEASURED_BANDS_COL
).round(3)

**Empfehlung:** Die Regel wählt **Sentinel-2** (10 gemessene Bänder, CV 0,852). Sie ist damit auch die praktisch verfügbare Variante, eine zweite Variante entfällt. Validiert werden Sentinel-2 und die Referenz mit allen Bändern.

## 5. Einmalige Validierung

Die gewählten Varianten werden mit ihren besten CV-Einstellungen **einmal** auf dem Validierungsteil bewertet. Danach wird nichts mehr eingestellt.

In [ ]:
rows = curve.set_index(REDUCTION_VARIANT_COL)
by_name = {variant.name: variant for variant in variants}
results = {}
for name in chosen:
    results[name] = run(
        make_combined_svm().set_params(**rows.loc[name, REDUCTION_BEST_PARAMS_COL]),
        RunConfig(
            name=f"reduction_svm_{name}",
            approach=APPROACH,
            experiment=MLFLOW_REDUCTION_EXPERIMENT,
            description=f"Datenreduktion, Variante {name}, beste CV-Einstellung",
            tuning_run=rows.loc[name, REDUCTION_TUNING_RUN_COL],
            preprocessing=by_name[name].preprocessing,
        ),
    )
pd.DataFrame(
    {
        name: {
            **result.metrics.model_dump(),
            **result.input_size._asdict(),
            "run_id": result.run_id[:8],
        }
        for name, result in results.items()
    }
).T

In [ ]:
validation_ax = plot_metric_comparison({name: result.metrics for name, result in results.items()})
validation_ax.set_title("Validierung: alle Bänder vs. reduzierte Eingaben")
save_doc_figure(validation_ax.figure, "datenreduktion_validierung", MODEL_DOCS_FIGURES_DIR)

## Fazit

| | Gemessene Bänder | CV kombiniert | Val. Kultur | Val. Stadium | Val. kombiniert |
| --- | --- | --- | --- | --- | --- |
| Alle Bänder (`706e5478`) | 131 | 0,848 | 0,925 | 0,901 | 0,859 |
| Sentinel-2 (`df2f00d0`) | 10 | 0,852 | 0,927 | 0,898 | 0,859 |

- **Kein Qualitätsverlust bei 92 % weniger Bändern:** 10 breite Sentinel-2-Bänder plus AEZ und Monat erreichen auf der Validierung dieselbe Qualität wie alle 131 Hyperion-Bänder. Das bestätigt die vorab festgelegte CV-Regel.
- **Für die Frage, welche Daten nötig sind:**
    - Wichtiger als eine hohe spektrale Auflösung ist, dass alle Bereiche von VIS bis SWIR2 abgedeckt sind.
    - AEZ und Monat lassen sich nicht ersetzen.
    - Indizes, Albedo und PCA bringen dem Modell keinen Mehrwert.
- **Einschränkungen:**
    - Die Sentinel-2-Simulation ist idealisiert: Rechteck-Mittelung statt der echten spektralen Empfindlichkeit, dieselben Aufnahmen, keine gröbere räumliche Auflösung. Mit echten Sentinel-2-Daten wäre die Qualität vermutlich etwas niedriger.
    - Geprüft ist nur die kombinierte SVM. Der hierarchische Ansatz folgt in #163.
- **Nächste Schritte:**
    - Prüfen, ob der zufällige Split durch gemeinsame Aufnahmen (AEZ × Monat) zu optimistisch ist.
    - Den Zeitbezug (Monat) als zweite Reduktionsachse untersuchen.
    - Feature-Importance (#156): welche Sentinel-2-Bänder tragen.